In [1]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from typing import Tuple
from PIL import Image
import random

# ─────────────────────────────────────────────────────────
# 1. DATASET (Гарантована бінаризація маски)
# ─────────────────────────────────────────────────────────
class VesuviusInkDataset(Dataset):
    def __init__(self, surface_volume_dir, ink_mask_path,
                 crop_size=(128, 128), stride=(128, 128),
                 augment=True, max_chunks=None):
        self.crop_size = crop_size
        self.stride    = stride
        self.augment   = augment

        self.ct_files = sorted(glob.glob(os.path.join(surface_volume_dir, "*.tif")))
        if len(self.ct_files) == 0:
            raise ValueError(f"Не знайдено z-зрізів у {surface_volume_dir}")

        # ВИПРАВЛЕНО: Конвертуємо в 'L', бінаризуємо в {0, 1}
        ink = np.array(Image.open(ink_mask_path).convert('L'))
        if ink.mean() > 127: ink = 255 - ink
        self.ink_mask = (ink > 127).astype(np.uint8)
        
        # ВАЛІДАЦІЯ: Код зупиниться тут, якщо маска не бінарна
        assert self.ink_mask.max() == 1, f"Маска не бінарна! Max={self.ink_mask.max()}"
        
        self.H, self.W = self.ink_mask.shape
        self.D         = len(self.ct_files)

        ink_ratio = self.ink_mask.mean()
        pos_weight = (1 - ink_ratio) / (ink_ratio + 1e-6)
        self.pos_weight = min(pos_weight, 5.0) # Обмежуємо вагу
        print(f"✅ Mask OK: Sum={self.ink_mask.sum()}, Ratio={ink_ratio:.4f}, PosWeight={self.pos_weight:.2f}")

        print(f"Кешування volume ({self.D} зрізів × {self.H}×{self.W}) як uint8...")
        slices = []
        for f in self.ct_files:
            try:
                img = np.array(Image.open(f))
                if len(img.shape) == 3: img = img[:, :, 0]
                slices.append(img) 
            except Exception:
                slices.append(np.zeros((self.H, self.W), dtype=np.uint8))
        self.volume = np.stack(slices, axis=0)  
        print(f"✓ Volume кешовано: {self.volume.shape}, RAM ≈ {self.volume.nbytes // 1024**2} MB")

        self.chunks = []
        for y in range(0, self.H - crop_size[0] + 1, stride[0]):
            for x in range(0, self.W - crop_size[1] + 1, stride[1]):
                self.chunks.append((y, x))
                    
        random.seed(42)
        random.shuffle(self.chunks)
        if max_chunks: self.chunks = self.chunks[:max_chunks]

    def __len__(self): return len(self.chunks)

    def __getitem__(self, idx):
        y, x = self.chunks[idx]
        h, w = self.crop_size

        y1, y2 = y, min(y + h, self.H)
        x1, x2 = x, min(x + w, self.W)

        ct_crop = np.zeros((self.D, h, w), dtype=np.float32)
        ct_crop[:, :y2-y1, :x2-x1] = self.volume[:, y1:y2, x1:x2].astype(np.float32)
        
        p1, p99 = np.percentile(ct_crop, [1, 99])
        ct_crop = np.clip(ct_crop, p1, p99)
        ct_crop = (ct_crop - p1) / (p99 - p1 + 1e-8)
        ct_crop = ct_crop.astype(np.float32)

        mask_crop = np.zeros((h, w), dtype=np.int64)
        mask_crop[:y2-y1, :x2-x1] = self.ink_mask[y1:y2, x1:x2]

        if self.augment:
            if np.random.rand() > 0.5:
                ct_crop   = np.flip(ct_crop,   axis=1).copy()
                mask_crop = np.flip(mask_crop, axis=0).copy()
            if np.random.rand() > 0.5:
                ct_crop   = np.flip(ct_crop,   axis=2).copy()
                mask_crop = np.flip(mask_crop, axis=1).copy()

        return torch.from_numpy(ct_crop), torch.from_numpy(mask_crop)

# ─────────────────────────────────────────────────────────
# 2. MODEL (SUBIT-Ink 2D Architecture)
# ─────────────────────────────────────────────────────────
class CurvatureField2D(nn.Module):
    def __init__(self, sigma: float = 1.0):
        super().__init__()
        self.smooth = self._make_gaussian_kernel(sigma)

    def _make_gaussian_kernel(self, sigma: float) -> nn.Conv2d:
        size = int(6 * sigma + 1) | 1
        coords = torch.arange(size).float() - size // 2
        g = torch.exp(-coords**2 / (2 * sigma**2))
        g = g / g.sum()
        kernel = (g[:, None] * g[None, :]).unsqueeze(0).unsqueeze(0)
        conv = nn.Conv2d(1, 1, size, padding=size//2, bias=False)
        conv.weight = nn.Parameter(kernel, requires_grad=False)
        return conv

    def _second_derivative(self, x: torch.Tensor, dim: int) -> torch.Tensor:
        pad = (1, 1, 0, 0) if dim == 3 else (0, 0, 1, 1)
        xp = F.pad(x, pad, mode='replicate')
        sm = [slice(None)] * xp.dim(); sc = [slice(None)] * xp.dim(); sp = [slice(None)] * xp.dim()
        sm[dim] = slice(0, -2); sc[dim] = slice(1, -1); sp[dim] = slice(2, None)
        return xp[tuple(sm)] - 2 * xp[tuple(sc)] + xp[tuple(sp)]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xg = x.mean(dim=1, keepdim=True)
        xs = self.smooth(xg)
        hyy = self._second_derivative(xs, 2)
        hxx = self._second_derivative(xs, 3)
        l1 = torch.min(hxx.abs(), hyy.abs())
        l2 = torch.max(hxx.abs(), hyy.abs())
        norm = l2 + 1e-6
        return torch.cat([1 - l1 / norm, l1 / norm, ((hxx < 0) & (hyy < 0)).float(), ((hxx > 0) & (hyy > 0)).float()], dim=1)

class SUBITv2Encoder2D(nn.Module):
    WHEN_SCALES = [1, 2, 4, 8]
    def __init__(self, embed_dim: int = 32):
        super().__init__()
        self.curvature_fields = nn.ModuleList([CurvatureField2D(sigma=s) for s in self.WHEN_SCALES])
        self.subit_proj = nn.Sequential(
            nn.Conv2d(16, embed_dim // 2, 1), nn.GroupNorm(8, embed_dim // 2), nn.GELU(),
            nn.Conv2d(embed_dim // 2, embed_dim, 1),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.subit_proj(torch.cat([cf(x) for cf in self.curvature_fields], dim=1))

class LearnedEvolutionFtheta2D(nn.Module):
    def __init__(self, channels: int, n_steps: int = 3):
        super().__init__()
        self.n_steps = n_steps
        self.diffusion_net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels),
            nn.Conv2d(channels, channels, 1), nn.GroupNorm(8, channels), nn.GELU(),
            nn.Conv2d(channels, 2, 1),
        )
        self.dt = nn.Parameter(torch.ones(1, channels, 1, 1) * 0.1)
        self.proj = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.GELU())
        self.stability_gate = nn.Sequential(nn.Conv2d(channels, 1, 1), nn.Sigmoid())

    def anisotropic_step(self, x: torch.Tensor, D_tensor: torch.Tensor) -> torch.Tensor:
        Dxx, Dyy = D_tensor[:, 0:1].abs(), D_tensor[:, 1:2].abs()
        xp = nn.ReplicationPad2d(1)(x)
        d2y = xp[:, :, 2:, 1:-1] - 2 * x + xp[:, :, :-2, 1:-1]
        d2x = xp[:, :, 1:-1, 2:] - 2 * x + xp[:, :, 1:-1, :-2]
        return x + self.dt * (Dyy * d2y + Dxx * d2x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual, D_tensor, gate, h = x, self.diffusion_net(x), self.stability_gate(x), x
        for _ in range(self.n_steps): h = self.anisotropic_step(h, D_tensor)
        return residual + gate * (self.proj(h) - residual)

class ConvBlock2D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.GroupNorm(min(8, out_ch), out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(min(8, out_ch), out_ch), nn.GELU(),
        )
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x): return self.block(x) + self.skip(x)

class SUBITAttention2D(nn.Module):
    def __init__(self, feat_ch: int, subit_ch: int):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Conv2d(feat_ch + subit_ch, feat_ch, 1), nn.GroupNorm(min(8, feat_ch), feat_ch), nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 1), nn.Sigmoid(),
        )
    def forward(self, feat, subit): return feat * self.gate(torch.cat([feat, subit], dim=1))

class SUBITInkUNet(nn.Module):
    def __init__(self, in_channels: int = 65, features: Tuple[int, ...] = (32, 64, 128, 256), subit_embed_dim: int = 32):
        super().__init__()
        self.subit_encoder = SUBITv2Encoder2D(embed_dim=subit_embed_dim)
        self.input_proj    = nn.Conv2d(in_channels + subit_embed_dim, features[0], 1)
        self.encoders, self.pools, self.enc_attn = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        ch = features[0]
        for f in features:
            self.encoders.append(ConvBlock2D(ch, f)); self.pools.append(nn.MaxPool2d(2))
            self.enc_attn.append(SUBITAttention2D(f, subit_embed_dim)); ch = f
        self.bottleneck = ConvBlock2D(features[-1], features[-1] * 2)
        self.evolution  = LearnedEvolutionFtheta2D(features[-1] * 2, n_steps=3)
        self.upconvs, self.decoders, self.dec_attn = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        ch = features[-1] * 2
        for f in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(ch, f, 2, stride=2))
            self.decoders.append(ConvBlock2D(f * 2, f)); self.dec_attn.append(SUBITAttention2D(f, subit_embed_dim)); ch = f
        self.feature_proj = nn.Conv2d(features[0], features[0], 1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        subit_emb = self.subit_encoder(x)
        subit_skips = [subit_emb]
        for _ in range(len(self.encoders) - 1): subit_skips.append(F.avg_pool2d(subit_skips[-1], 2))
        h = self.input_proj(torch.cat([x, subit_emb], dim=1))
        skips = []
        for i, (enc, pool, attn) in enumerate(zip(self.encoders, self.pools, self.enc_attn)):
            h = attn(enc(h), subit_skips[i]); skips.append(h); h = pool(h)
        h = self.evolution(self.bottleneck(h))
        for upconv, dec, attn, skip in zip(self.upconvs, self.decoders, self.dec_attn, reversed(skips)):
            h = upconv(h)
            if h.shape != skip.shape: h = F.interpolate(h, size=skip.shape[2:], mode='bilinear', align_corners=False)
            h = attn(dec(torch.cat([h, skip], dim=1)), F.interpolate(subit_emb, size=h.shape[2:], mode='bilinear', align_corners=False))
        return self.feature_proj(h), subit_emb

class AttractorDecoder2D(nn.Module):
    def __init__(self, channels: int, n_attractor_steps: int = 5):
        super().__init__()
        self.n_steps = n_attractor_steps
        self.ftheta = LearnedEvolutionFtheta2D(channels, n_steps=1)
        self.omega_head = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1), nn.GroupNorm(min(8, channels), channels), nn.GELU(),
            nn.Conv2d(channels, 4, 1), nn.Softmax(dim=1),
        )
        self.ink_head = nn.Conv2d(channels, 2, 1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h, delta = x, torch.zeros_like(x)
        for _ in range(self.n_steps):
            h_new = self.ftheta(h)
            delta += (h_new - h).abs()
            h = h_new
        omega = self.omega_head(torch.cat([h, delta], dim=1))
        omega_weight = omega[:, 3:4] * 1.5 + omega[:, 2:3] * 1.2 + omega[:, 1:2] * 0.8 + omega[:, 0:1] * 0.3
        return self.ink_head(h * omega_weight), omega

class OmegaLossV2_2D(nn.Module):
    def __init__(self, pos_weight=5.0, alpha=0.4, beta=0.3, gamma=0.2, delta=0.1):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.delta = alpha, beta, gamma, delta
        self.register_buffer('weight', torch.tensor([1.0, float(pos_weight)], dtype=torch.float32))

    def forward(self, logits, labels, omega, subit_emb):
        probs = F.softmax(logits, dim=1)
        ce = F.cross_entropy(logits, labels, weight=self.weight)
        
        smooth = 1.0
        p, l = probs[:, 1], labels.float()
        inter = (p * l).sum(dim=(1, 2))
        union = p.sum(dim=(1, 2)) + l.sum(dim=(1, 2))
        dice = (1 - (2 * inter + smooth) / (union + smooth)).mean()
        
        if subit_emb.shape[2:] != probs.shape[2:]:
            subit_emb = F.interpolate(subit_emb, size=probs.shape[2:], mode='bilinear', align_corners=False)
        curv = (torch.sigmoid(subit_emb[:, 0]) * (1 - probs[:, 1])).mean()
        stab = (omega[:, 0] * probs[:, 1]).mean()
        omega_mean = omega.mean(dim=(0, 2, 3))
        omega_reg = (omega_mean * (omega_mean + 1e-8).log()).sum() * 0.01
        total = self.alpha * ce + (1 - self.alpha) * dice + self.gamma * curv + self.beta * stab + self.delta * omega_reg
        return total, {"ce": ce.item(), "dice": dice.item(), "curvature": curv.item(), "stability": stab.item(), "omega_reg": omega_reg.item(), "total": total.item()}

class VesuviusInkPipeline:
    def __init__(self, device: str = 'cuda', lr: float = 5e-5, pos_weight=5.0):
        self.device = device
        self.model = SUBITInkUNet(in_channels=65).to(device)
        self.attractor = AttractorDecoder2D(channels=32).to(device)
        self.loss_fn = OmegaLossV2_2D(pos_weight=pos_weight).to(device)
        self.optimizer = torch.optim.AdamW(list(self.model.parameters()) + list(self.attractor.parameters()), lr=lr, weight_decay=1e-5)

    def train_step_batch(self, ct_batch, mask_batch):
        self.model.train(); self.attractor.train(); self.optimizer.zero_grad()
        x = ct_batch.float().to(self.device)
        y = mask_batch.long().to(self.device)
        
        features, subit_emb = self.model(x)
        logits, omega = self.attractor(features)
        if logits.shape[2:] != y.shape[1:]:
            logits = F.interpolate(logits, size=y.shape[1:], mode='bilinear', align_corners=False)
            omega = F.interpolate(omega, size=y.shape[1:], mode='bilinear', align_corners=False)
        loss, components = self.loss_fn(logits, y, omega, subit_emb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 0.5)
        self.optimizer.step()
        with torch.no_grad():
            pred = (F.softmax(logits, dim=1)[:, 1] > 0.5).long()
            p, t = pred.float().flatten(), y.float().flatten()
            components["dice_metric"] = ((2 * (p * t).sum() + 1e-5) / (p.sum() + t.sum() + 1e-5)).item()
        return components

    def val_step_batch(self, ct_batch, mask_batch):
        self.model.eval(); self.attractor.eval()
        with torch.no_grad():
            x = ct_batch.float().to(self.device)
            y = mask_batch.long().to(self.device)
            
            features, subit_emb = self.model(x)
            logits, omega = self.attractor(features)
            if logits.shape[2:] != y.shape[1:]:
                logits = F.interpolate(logits, size=y.shape[1:], mode='bilinear', align_corners=False)
                omega = F.interpolate(omega, size=y.shape[1:], mode='bilinear', align_corners=False)
            loss, components = self.loss_fn(logits, y, omega, subit_emb)
            pred = (F.softmax(logits, dim=1)[:, 1] > 0.5).long()
            p, t = pred.float().flatten(), y.float().flatten()
            components["dice_metric"] = ((2 * (p * t).sum() + 1e-5) / (p.sum() + t.sum() + 1e-5)).item()
        return components

    def save(self, path: str):
        torch.save({'model_state': self.model.state_dict(), 'attractor_state': self.attractor.state_dict()}, path)

# ─────────────────────────────────────────────────────────
# 3. FULL TRAINING & SUBMISSION
# ─────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print("🚀 SUBIT-Ink v1.0 — FULL KAGGLE RUN (Strict Binary Mask)")
    print("=" * 60)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    base_path = "/kaggle/input/competitions/vesuvius-challenge-ink-detection"
    train_path = os.path.join(base_path, "train", "1")
    save_path = "/kaggle/working/subit_ink_best.pt"
    
    # --- КРОК 1: ТРЕНУВАННЯ ---
    print("\n[1/2] Запуск тренування (10 епох)...")
    full_ds = VesuviusInkDataset(os.path.join(train_path, "surface_volume"), os.path.join(train_path, "inklabels.png"), crop_size=(128, 128), stride=(128, 128), augment=True, max_chunks=None)
    
    val_n = int(len(full_ds) * 0.2)
    train_n = len(full_ds) - val_n
    
    train_ds = VesuviusInkDataset.__new__(VesuviusInkDataset)
    train_ds.__dict__.update(full_ds.__dict__)
    train_ds.chunks = full_ds.chunks[:train_n]; train_ds.augment = True
    
    val_ds = VesuviusInkDataset.__new__(VesuviusInkDataset)
    val_ds.__dict__.update(full_ds.__dict__)
    val_ds.chunks = full_ds.chunks[train_n:]; val_ds.augment = False
    
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2)
    
    pipeline = VesuviusInkPipeline(device=device, pos_weight=full_ds.pos_weight)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(pipeline.optimizer, mode='max', factor=0.5, patience=2)
    
    EPOCHS = 10
    best_val_dice = 0.0
    for epoch in range(EPOCHS):
        train_loss, train_dice = 0.0, 0.0
        pbar = tqdm(train_loader, desc=f"Train E{epoch+1}/{EPOCHS}")
        for ct, mask in pbar:
            m = pipeline.train_step_batch(ct, mask)
            train_loss += m['total']; train_dice += m['dice_metric']
            pbar.set_postfix({'Loss': f"{m['total']:.4f}", 'Dice': f"{m['dice_metric']:.4f}"})
            
        val_loss, val_dice = 0.0, 0.0
        for ct, mask in tqdm(val_loader, desc=f"Val E{epoch+1}/{EPOCHS}"):
            m = pipeline.val_step_batch(ct, mask)
            val_loss += m['total']; val_dice += m['dice_metric']
            
        avg_vd = val_dice / len(val_loader)
        print(f"Epoch {epoch+1}: Val Dice: {avg_vd:.4f}")
        
        scheduler.step(avg_vd)
        if avg_vd > best_val_dice:
            best_val_dice = avg_vd
            pipeline.save(save_path)
            print(f"  ✅ Best model saved!")
            
        torch.cuda.empty_cache()

    del full_ds, train_ds, val_ds, train_loader, val_loader
    import gc; gc.collect()

    # --- КРОК 2: ІНФЕРЕНС ТА СУБМІШН ---
    print("\n[2/2] Генерація submission.csv...")
    ckpt = torch.load(save_path, map_location=device)
    pipeline.model.load_state_dict(ckpt['model_state'])
    pipeline.attractor.load_state_dict(ckpt['attractor_state'])
    pipeline.model.eval(); pipeline.attractor.eval()
    
    test_dir = os.path.join(base_path, "test")
    sub_data = []
    
    for frag in ["a", "b"]:
        frag_path = os.path.join(test_dir, frag, "surface_volume")
        if not os.path.exists(frag_path): continue
        
        files = sorted(glob.glob(os.path.join(frag_path, "*.tif")))
        
        print(f"Завантаження тестового фрагмента {frag}...")
        test_slices = [np.array(Image.open(f)) for f in files]
        test_vol = np.stack(test_slices, axis=0) 
        H, W = test_vol.shape[1], test_vol.shape[2]
        D = test_vol.shape[0]
        
        total_probs = np.zeros((H, W), dtype=np.float32)
        counts = np.zeros((H, W), dtype=np.uint8)
        
        CROP_H, CROP_W = 128, 128
        STRIDE = 64
        
        for y in tqdm(range(0, H, STRIDE), desc=f"Frag {frag} Infer"):
            for x in range(0, W, STRIDE):
                y_end, x_end = min(y + CROP_H, H), min(x + CROP_W, W)
                
                ct_crop = np.zeros((D, CROP_H, CROP_W), dtype=np.float32)
                ct_crop[:, :y_end-y, :x_end-x] = test_vol[:, y:y_end, x:x_end].astype(np.float32)
                
                p1, p99 = np.percentile(ct_crop, [1, 99])
                ct_crop = np.clip(ct_crop, p1, p99)
                ct_crop = (ct_crop - p1) / (p99 - p1 + 1e-8)
                ct_crop = ct_crop.astype(np.float32)
                
                x_tensor = torch.from_numpy(ct_crop).unsqueeze(0).to(device)
                with torch.no_grad():
                    features, _ = pipeline.model(x_tensor)
                    logits, _ = pipeline.attractor(features)
                    probs = F.softmax(logits, dim=1)[0, 1].cpu().numpy()
                    
                total_probs[y:y_end, x:x_end] += probs[:y_end-y, :x_end-x]
                counts[y:y_end, x:x_end] += 1

        del test_vol, test_slices; gc.collect()

        avg_probs = total_probs / (counts + 1e-8)
        final_mask = (avg_probs > 0.5).astype(np.uint8)
        
        pixels = final_mask.flatten()
        pixels = np.concatenate([[0], pixels, [0]])
        runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
        runs[1::2] -= runs[::2]
        rle = ' '.join(str(x) for x in runs)
        sub_data.append({'Id': frag, 'Predicted': rle})
        
    sub_df = pd.DataFrame(sub_data)
    sub_df.to_csv('/kaggle/working/submission.csv', index=False)
    print("\n✅ Успіх! Файл submission.csv створено.")

🚀 SUBIT-Ink v1.0 — FULL KAGGLE RUN (Strict Binary Mask)

[1/2] Запуск тренування (10 епох)...
✅ Mask OK: Sum=5339362, Ratio=0.1031, PosWeight=5.00
Кешування volume (65 зрізів × 8181×6330) як uint8...
✓ Volume кешовано: (65, 8181, 6330), RAM ≈ 6420 MB


Val E1/10: 100%|██████████| 155/155 [00:09<00:00, 16.72it/s]


Epoch 1: Val Dice: 0.2124
  ✅ Best model saved!


Val E2/10: 100%|██████████| 155/155 [00:08<00:00, 17.77it/s]


Epoch 2: Val Dice: 0.0958


...

Epoch 10: Val Dice: 0.3630

[2/2] Генерація submission.csv...
Завантаження тестового фрагмента a...


Frag a Infer: 100%|██████████| 43/43 [02:16<00:00,  3.17s/it]


Завантаження тестового фрагмента b...


Frag b Infer: 100%|██████████| 86/86 [04:25<00:00,  3.09s/it]



✅ Успіх! Файл submission.csv створено.
